# Multilayer Perceptron (MLP) from Scratch with NumPy

In [ ]:
#!pip install git+https://github.com/UBod/pyMLaux.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Optional
from pyMLaux import plot_2d_prediction

## Utilities


In [ ]:
def one_hot(y: np.ndarray, num_classes: int) -> np.ndarray:
    oh = np.zeros((y.size, num_classes))
    oh[np.arange(y.size), y] = 1.0
    return oh

# Initialization helpers

# Activations and their derivatives

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def dsigmoid(z):
    s = sigmoid(z)
    return s * (1 - s)

# Stable softmax + cross-entropy

def softmax(logits: np.ndarray) -> np.ndarray:
    z = logits - logits.max(axis=1, keepdims=True)
    expz = np.exp(z)
    return expz / expz.sum(axis=1, keepdims=True)

def binary_cross_entropy(yhat: float, y_true: float, eps: float = 1e-12) -> float:
    yhat_clipped = np.clip(yhat, eps, 1.0)
    return -(y_true * np.log(yhat_clipped) + (1 - y_true) * np.log(1 - yhat_clipped))

def cross_entropy(probs: np.ndarray, y_true_oh: np.ndarray, eps: float = 1e-12) -> float:
    # Avoid log(0)
    p = np.clip(probs, eps, 1.0)
    return -np.mean(np.sum(y_true_oh * np.log(p), axis=1))


# Basic Example (One hidden neuron)

In [ ]:
# Learning rate
lr = 0.1

# One data point (x, y)
x = np.array([0.5, 0.7])   # shape (2,)
y = 1.0

# Parameters: hidden (1 neuron) + output (1 neuron)
w1 = np.array([[0.3, -0.5],
               [0.7, 0.1]])  # shape (2, 2)

b1 = np.array([1.2, -0.6])        # shape (2,)
w2 = np.array([0.8, 0.3]) 
b2 = -0.4

In [ ]:
# ----- Forward -----
z1 = w1 @ x + b1
a1 = sigmoid(z1)
z2 = w2 @ a1 +b2
yhat = a2 = sigmoid(z2)
loss = binary_cross_entropy(yhat, y)

# ----- Backward (analytic) -----
dz2 = yhat - y
dw2 = dz2 * a1
db2 = dz2



# ----- Weight Adjustment -----

##  MLP

In [ ]:
class MLP:
    def __init__(
        self,
        input_dim: int,
        layer_sizes: List[int],
        num_classes: int,
        seed: Optional[int] = None,
    ):
        self.rng = np.random.default_rng(seed)
        self.shapes = []

        self.act = sigmoid
        self.dact = dsigmoid

        # layer dimensions
        dims = [input_dim] + layer_sizes + [num_classes]
        self.L = len(dims) - 1

        # params init
        self.W = {}
        self.b = {}
        for k in range(1, self.L + 1):
            fan_in, fan_out = dims[k-1], dims[k]
            Wk = np.random.randn(fan_in, fan_out)
            bk = np.zeros((1, fan_out))
            self.W[k] = Wk
            self.b[k] = bk
            self.shapes.append((fan_in, fan_out))

    def forward(self, X: np.ndarray, train: bool = True):
        pass

    def backward(self, probs: np.ndarray, y_onehot: np.ndarray, caches: List[tuple]):
        pass

    def step(self, grads_W: Dict[int, np.ndarray], grads_b: Dict[int, np.ndarray], lr: float): 
        pass

    def compute_loss(self, probs: np.ndarray, y_onehot: np.ndarray):
        loss = cross_entropy(probs, y_onehot)
        return loss

    def predict(self, X: np.ndarray):
        probs, _ = self.forward(X, train=False)
        return np.argmax(probs, axis=1), probs


## 4. Training Loop

In [ ]:
def accuracy(y_true, y_pred):
    return (y_true == y_pred).mean()


def train(
    model: MLP,
    X: np.ndarray,
    y: np.ndarray,
    epochs: int = 200,
    lr: float = 1e-0,
    seed: Optional[int] = 123,
    verbose: bool = True
):
    np.random.default_rng(seed)
    num_classes = int(y.max()) + 1
    y_oh = one_hot(y, num_classes)
    history = {"loss": [], "acc": []}

    for epoch in range(1, epochs + 1):
        # Training epoch

        r = np.arange(X.shape[0])
        np.random.shuffle(r)
        for i in r:
            #Update weights with gradients
            pass

        # Evaluate on full training set (for simplicity)
        probs_full, _ = model.forward(X, train=False)
        loss = model.compute_loss(probs_full, y_oh)
        y_pred = np.argmax(probs_full, axis=1)
        acc = accuracy(y, y_pred)
        history["loss"].append(loss)
        history["acc"].append(acc)

        if verbose and (epoch % max(1, epochs // 10) == 0 or epoch <= 5):
            print(f"Epoch {epoch:4d} | loss={loss:.4f} | acc={acc*100:.2f}%")
    return history


## Train a model for nonseparable data

In [ ]:
X = np.array([[0.1, 0.5], [0.2, 0.9], [0.4, 0.7], [0.5, 0.8],
              [0.4, 0.2], [0.6, 0.3], [0.7, 0.05], [0.8, 0.5]])
y = np.array([1, 1, 1, 1, 0, 1, 0, 0])

# Hyperparameters
input_dim = X.shape[1]
num_classes = int(y.max()) + 1
layer_sizes = [10,10]

model = MLP(
    input_dim=input_dim,
    layer_sizes=layer_sizes,
    num_classes=num_classes,
    seed=42,
)

history = train(
    model, X, y,
    epochs=3000,
    lr=0.2,
    seed=123,
)

plot_2d_prediction(X, y, lambda x: model.predict(x)[0],
                   xval=np.linspace(-0.1, 1.1, 101), yval=np.linspace(-0.1, 1.1, 101))

In [ ]:
#plot loss and accuracy
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history['loss'])
plt.title('Loss'); plt.xlabel('epoch'); plt.ylabel('loss')
plt.subplot(1,2,2)
plt.plot(history['acc'])
plt.title('Accuracy'); plt.xlabel('epoch'); plt.ylabel('acc')
plt.show()

## Train a model for spiral data

In [ ]:
#Sprial Dataset generator
def make_spiral(n_per_class: int = 200, num_classes: int = 3, noise: float = 0.2, seed: int = 42):
    rng = np.random.default_rng(seed)
    D = 2  # 2D inputs for easy visualization
    X = np.zeros((n_per_class * num_classes, D))
    y = np.zeros(n_per_class * num_classes, dtype=np.int64)

    for j in range(num_classes):
        ix = range(j * n_per_class, (j + 1) * n_per_class)
        r = np.linspace(0.0, 1, n_per_class)  # radius
        t = np.linspace(j * 4, (j + 1) * 4, n_per_class) + rng.normal(0, noise, n_per_class)  # theta
        X[ix] = np.c_[r * np.sin(t), r * np.cos(t)]
        y[ix] = j
    return X, y

X, y = make_spiral(n_per_class=200, num_classes=3, noise=0.2, seed=1)
print(X.shape, y.shape)

# quick plot
plt.figure(figsize=(5,5))
plt.scatter(X[:,0], X[:,1], c=y, cmap='viridis', s=10)
plt.title('Spiral dataset')
plt.show()

In [ ]:
# Hyperparameters
input_dim = X.shape[1]
num_classes = int(y.max()) + 1
layer_sizes = [10,10]

model = MLP(
    input_dim=input_dim,
    layer_sizes=layer_sizes,
    num_classes=num_classes,
    seed=42,
)

history = train(
    model, X, y,
    epochs=200,
    lr=0.2,
    seed=123,
)


## 6. Decision Boundary Visualization
Plot the learned decision regions in 2D.


In [ ]:
def plot_decision_boundary(model: MLP, X: np.ndarray, y: np.ndarray, h: float = 0.01, inter = 0):
    x_min, x_max = X[:, 0].min() - 0.2, X[:, 0].max() + 0.2
    y_min, y_max = X[:, 1].min() - 0.2, X[:, 1].max() + 0.2
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    preds, _ = model.predict(grid)
    Z = preds.reshape(xx.shape)

    plt.figure(figsize=(6,6))
    plt.contourf(xx, yy, Z, cmap='viridis', alpha=0.25, levels=np.arange(-0.5, int(y.max())+1.5, 1))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=10, edgecolor='k', linewidth=0.2)
    plt.title('Decision boundary')

plot_decision_boundary(model, X, y)


In [ ]:
#plot loss and accuracy
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history['loss'])
plt.title('Loss'); plt.xlabel('epoch'); plt.ylabel('loss')
plt.subplot(1,2,2)
plt.plot(history['acc'])
plt.title('Accuracy'); plt.xlabel('epoch'); plt.ylabel('acc')
plt.show()


# 1. Mini Batches
To improve the learning stability, introduce batched learning. (See Unit 3 Page 95 of the lecture for guidance) 

# 2. Work with real dataset
Try to use your updated multi layer perceptron algorithm to fit and predict the penguin data set. Don't forget to preprocess the data accordingly. Play around with the sizes of the model to achieve good results.